# STIC model validation

Validation of the model using field data from the ameriflux database and landsat acquisitions

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
import datetime as dt
from collections import deque
from itertools import repeat

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from numba import njit
from plotly.subplots import make_subplots
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sweat.stic.functions import convert_kelvin_to_celsius, convert_to_rh, convert_to_local_time, CST_KELVIN, CST_SB, _tetens
from sweat.stic import main as stic
from sweat.common.flux import compute_et_from_le
from sweat.common.solar import compute_toa_solar_radiation, compute_daily_toa_solar_radiation

In [ ]:
if os.environ.get("SWEAT_TEST_DATA_PATH", None) is None:
    msg = "Variable SWEAT_TEST_DATA_PATH must be set"
    raise ValueError(msg)

## Input

In [ ]:
stic_data_path = os.path.join(os.environ.get("SWEAT_TEST_DATA_PATH"),"stic")
input_data_path = os.path.join(stic_data_path,"California_2017-2025_PointScale.xlsx")

## Get information on input data

In [ ]:
def get_info(filename:str)-> pd.DataFrame:
    """
    Get information
    """
    return pd.read_excel(filename,sheet_name="INFO")

def read_input_data(filename:str) -> pd.DataFrame:
    """
    Read input data
    """
    sites = pd.ExcelFile(filename).sheet_names
    sites.remove("INFO")
    return pd.concat([df for _,df in pd.read_excel(filename,sheet_name=sites).items()])

def plot_data(df: pd.DataFrame,variable:str):
    """
    Plot data
    """
    fig = px.line(
        df,
        x="date",
        y=variable,
        color="name",   # group by "name"
        markers=True,
        custom_data=["name"],
    )

    # Print value
    fig.update_traces(
    hovertemplate=
    "<b>Site:</b> %{customdata[0]}<br>" +
    "<b>Date:</b> %{x|%Y-%m-%d}<br>" +
    f"<b>{variable}:</b> %{{y:.2f}}<br>" +
    "<extra></extra>"
    )

    # Format x-axis as YYYY-MM-DD
    fig.update_xaxes(
        tickformat="%Y-%m-%d"
    )

    # Improve layout
    fig.update_layout(
        xaxis_title="Date",
        yaxis_title=variable,
        legend_title="Site",
        title={
        "text": f"Evolution of {variable} over time",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 20}
       }
    )

    fig.show() 

Input content

In [ ]:
info_df = get_info(input_data_path)

In [ ]:
info_df

## Read data

In [ ]:
input_data = read_input_data(input_data_path)

In [ ]:
input_data

Get site names

In [ ]:
list(input_data["Name"].unique())

Extract data from one site

In [ ]:
bi1_input_data = input_data[input_data["Name"] == "US-Bi1_Alfalfa"]

In [ ]:
bi1_input_data.head()

## Process data

### Prepare data

* Convert temperatures to Celsius degrees.
* Compute local time in seconds taking into account the Landsat's overpass over the area around 18h45 UTC.
* Compute long wave net radiation with the following formula :
$$
L_{net} = \epsilon * \sigma_{SB} * 1.24 * (ea/(T_a+273.15))^{1/7}*(T_a+273.15)^4 - \epsilon*\sigma_{SB}*(LST+273.15)^4
$$
* Use **ssr** as net radiation

In [ ]:
def prepare_input_data(df:pd.DataFrame) -> pd.DataFrame:
    """
    Prepare data to run STIC
    """
    # ts, ta, td
    output_df = df[["st_b10","t2m","d2m"]].apply(convert_kelvin_to_celsius).rename(columns={"st_b10":"ts","t2m":"ta","d2m":"td"})
    # rh
    output_df["rh"] = output_df[["ta","td"]].apply(lambda row: convert_to_rh(row["ta"],row["td"]),axis=1)
    # fc
    output_df["fc"] = df["fcover"]
    # lai
    output_df["lai"] = df["lai"]
    # rn
    output_df["rn"] = df["ssr"]
    # ln
    eastar = output_df["ta"].apply(_tetens)
    ea = output_df["rh"] * eastar / 100
    output_df["ln"] = df["st_emis"] * CST_SB * 1.24 * (ea/(output_df["ta"]+CST_KELVIN))**(1/7)*(output_df["ta"]+CST_KELVIN)**4 - df["st_emis"]*CST_SB*(output_df["ts"]+CST_KELVIN)**4
    # localtime
    output_df["local_time"] = df[["Landsat_overpass","Lon","Lat"]].apply(lambda row: convert_to_local_time(date=dt.datetime.combine(row["Landsat_overpass"].date(),dt.time(hour=18,minute=45),tzinfo=dt.UTC),y=row["Lat"],x=row["Lon"],crs="epsg:4326"),axis=1)
    # name
    output_df["name"] = df["Name"]
    # lat/lon
    output_df["lon"] = df["Lon"]
    output_df["lat"] = df["Lat"]
    # dates
    output_df["date (utc)"] = df["Landsat_overpass"].apply(lambda t: dt.datetime.combine(t.date(),dt.time(hour=18,minute=45,tzinfo=dt.UTC)))
    output_df["date"] = output_df["date (utc)"].dt.tz_convert("America/Los_Angeles")
    # reorder columns
    return output_df[["name","date","date (utc)","lat","lon","ts","ta","td","rh","fc","lai","rn","ln","local_time"]]

def search_data(df:pd.DataFrame,name:str,date:str)-> pd.DataFrame:
    """ 
    Search data
    """ 
    date_dt = dt.datetime.strptime(date,"%Y-%m-%d")
    query = f"name.str.contains('{name}') and date >= '{date_dt:%Y-%m-%d} 00:00:00-07:00' and date<'{date_dt+dt.timedelta(days=1):%Y-%m-%d} 00:00:00-07:00' "
    return df.query(query)


In [ ]:
data = prepare_input_data(input_data)

In [ ]:
data.head()

### Plot data

In [ ]:
plot_data(data,variable="ts")

In [ ]:
plot_data(data,variable="ta")

In [ ]:
plot_data(data,variable="fc")

### Compute STIC model

* Compute LE, EF using STIC model for all input data
* Compute ET from LE

In [ ]:
@njit
def batch_run_stic_model(params:np.array):
    n = params.shape[0]
    out = np.empty((n, 2))  # only 2 outputs

    for i in range(n):
        le, ef, _ = stic.run_stic_model_pixel(
            ts=params[i,0], 
            ta=params[i,1], 
            td=params[i,2],
            rh=params[i,3],
            fc=params[i,4], 
            lai=params[i,5], 
            rn=params[i,6],
            ln=params[i,7],
            local_time=params[i,8],
            threshold=0.01,
            nb_steps=15,
            debug=False,
        )
        out[i,0] = le
        out[i,1] = ef

    return out

def run_stic(df: pd.DataFrame)-> pd.DataFrame:
    """
    Compute STIC model 
    """
    params_cols = ["ts","ta","td","rh","fc","lai","rn","ln","local_time"]
    params_arr = df[params_cols].to_numpy()
    res = batch_run_stic_model(params_arr)
    output_df = pd.DataFrame({
        "name": df["name"].values,
        "date": df["date"],
        "lat": df["lat"].values,
        "lon": df["lon"].values,
        "le": res[:,0],
        "ef": res[:,1],
    })
    output_df["et"] = output_df.apply(lambda x: compute_et_from_le(x["le"]),axis=1)
    return output_df

In [ ]:
estimated_data = run_stic(data)

In [ ]:
estimated_data

## Compare with EC data

* Data downloaded from ameriflux
* Use *LE_PI_F* or *LE_PI_F_1_1_1* for latent heat flux
* Use *H_PI_F* or *H_PI_F_1_1_1* for sensible heat flux
* Compute EF from LE and H
* Compute ET from LE
* Filter the data by Landsat acquisition dates 

In [ ]:
def get_code(name:str)-> str:
    """
    Get code site from the name
    """
    m = re.match(r'US-(.*)[_,-].*', name)
    if m:
        name = m.group(1)
    if len(name) >3:
        name = name[0:3]
    return name

def get_ec_data_filename(name:str, root:str)-> str:
    """
    Get EC data path for a site
    """
    code = get_code(name)
    files = [f for f in os.listdir(root) if re.match(r'AMF_US-(.*)_BASE_HH_.*\.csv', f)]
    for file in files:
        if code.lower() in file.lower():
            return os.path.join(root,file)
    raise IOError(f"File {file} for ground truth data not found")

def read_ec_data(name:str, root: str)-> pd.DataFrame:
    """
    Read EC data file for a site
    """
    filename = get_ec_data_filename(name,root)
    # Read
    df = pd.read_csv(filename,
                    sep=",",
                    encoding="ascii", 
                    skiprows=2, 
                    usecols=lambda col: col in ["TIMESTAMP_START","TIMESTAMP_END"] or col.startswith("H_PI_F") or col.startswith("LE_PI_F"),
                    parse_dates=["TIMESTAMP_START","TIMESTAMP_END"])
    # Rename columns
    df = df.rename(columns={"TIMESTAMP_START":"start","TIMESTAMP_END":"end"})
    best_le = min(
    (c for c in df.columns if c.startswith("LE_PI_F")),
    key=lambda c: (0,) if c == "LE_PI_F" else (1, *map(int, c.split("_")[3:]))
    )
    print(f"Site = {name} used measured LE = {best_le}")
    df = df.rename(columns={best_le: "ec_le"})
    best_h = min(
    (c for c in df.columns if c.startswith("H_PI_F")),
    key=lambda c: (0,) if c == "H_PI_F" else (1, *map(int, c.split("_")[3:]))
    )
    df = df.rename(columns={best_h: "ec_h"})
    # Exclude nodata
    df = df.mask(df.ec_le==-9999,np.nan).dropna()
    df = df.mask(df.ec_h==-9999,np.nan).dropna()
    # Compute EF
    df["ec_ef"] = df["ec_le"]/(df["ec_h"]+df["ec_le"])
    # Compute ET
    df["ec_et"] = df.apply(lambda x: compute_et_from_le(x["ec_le"]),axis=1)
    # Keep columns
    df = df[["start","end","ec_le","ec_ef","ec_et"]]
    return df

def get_ec_data(df:pd.DataFrame, path: str) -> pd.DataFrame:
    """
    Get EC data at acquisition dates
    """
    # Get sites
    sites = list(df["name"].unique())
    # Loop for each site
    datasets = []
    for site in sites:
        # Select data for a site
        selected_df = df[df["name"] == site].reset_index(drop=True)
        # Read
        ec_df = read_ec_data(site,path)
        # Search indexes
        intervals = pd.IntervalIndex.from_arrays(ec_df["start"].dt.tz_localize(None), ec_df["end"].dt.tz_localize(None), closed="both")
        matches,not_matches = intervals.get_indexer_non_unique(selected_df["date"].dt.tz_localize(None))
        # Remove no matches
        matches = matches[matches!=-1]
        datasets.append(pd.concat([selected_df.drop(not_matches).reset_index(drop=True),ec_df.iloc[matches].reset_index(drop=True)],axis=1))
    return pd.concat(datasets)
    

In [ ]:
results = get_ec_data(estimated_data, stic_data_path)

In [ ]:
results.head()

In [ ]:
results.tail()

### Compute metrics

Compute the following metrics for **latent heat flux LE**, **evapotransipration ET**, **evaporative fraction EF**:

* **Slope**: corresponds to the best-fit line forced through the origin and measures how the estimated ET data scale compared to the ground-based ET data. 
* **Mean Bias Error (MBE)**: quantifies the bias in the estimated ET data relative to the ground-based ET data
* **Mean Absolute Error (MAE)**: quantifies the average absolute difference between the estimated and ground-based ET data. 
* **Root Mean Squared Error (RMSE)**: quantifies the average absolute difference between the estimated and ground-based ET data, but provides greater weight to outliers or large ‘misses’ in the estimated ET data. 
* **r-squared**: provides a measure of the proportion of the variance in the ground-based ET data that is explained or reproduced by the estimated ET data. 


In [ ]:
def compute_metrics(measured: pd.Series, estimated: pd.Series)-> tuple[float,float,float,float,float]:
    """
    Compute slope, mbe, mae, rmse, r2
    """
    slope = np.polyfit(measured, estimated, 1)[0]
    mbe = np.mean(estimated-measured)
    mae = mean_absolute_error(measured, estimated)
    rmse = np.sqrt(mean_squared_error(measured, estimated))
    r2 = r2_score(measured, estimated)
    return (slope,mbe,mae,rmse,r2)
    
def generate_metrics_table(df:pd.DataFrame,variables:list[str]|None =None)->pd.DataFrame:
    """
    Generate metrics table
    """
    if variables is None:
        variables = ["le","et","ef"] 
    metrics = []
    # Compute by site
    for name,group in df.groupby("name"):
        for var in variables:
            slope, mbe, mae, rmse, r2 = compute_metrics(
                measured = group[f"ec_{var}"], 
                estimated=group[var]
                )
            metrics.append({
                "name":name,
                "variable":var,
                "slope": slope, 
                "mbe": mbe, 
                "mae":mae,
                "rmse":rmse,
                "r2":r2})
    # Compute for all
    for var in variables:
        slope, mbe, mae, rmse, r2 = compute_metrics(
            measured = group[f"ec_{var}"], 
            estimated=group[var])
        metrics.append({
               "name":"all",
                "variable":var,
                "slope": slope, 
                "mbe": mbe, 
                "mae":mae,
                "rmse":rmse,
                "r2":r2})
    return pd.DataFrame.from_records(metrics)

In [ ]:
generate_metrics_table(results)

In [ ]:
def plot_metrics(df:pd.DataFrame, variable:str="le"):
    """
    Plot metrics
    """
    groups = list(df.name.unique())
    groups.append("all")
    n_groups = len(groups)

    ncols = 3
    nrows = int(np.ceil(n_groups / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows))
    axes = axes.flatten()

    for i, group in enumerate(groups):

        if group == "all":
            subset = df
        else:
            subset = df[df.name == group]
            
        measured = subset[f"ec_{variable}"]
        estimated = subset[variable]

        ax = axes[i]
        ax.scatter(measured, estimated, s=14, alpha=0.5)

        lims = [min(measured.min(), estimated.min()), max(measured.max(), estimated.max())]
        ax.plot(lims, lims, "k--", lw=1)

        ax.set_aspect("equal", adjustable="box")
        ax.set_xlim(lims)
        ax.set_ylim(lims)

        ax.set_title(group)
        ax.set_xlabel("Measured")
        ax.set_ylabel("Estimated")

        # Calculate metrics
        slope, mbe, mae, rmse, r2 = compute_metrics(measured, estimated)

        # Plot linear fit
        slope, intercept = np.polyfit(measured, estimated, 1)
        ax.plot([0,measured.max()],[intercept,measured.max()*slope+intercept],"r-",lw=1)
        # Print metrics
        ax.text(
            0.97, 0.03,
            (
                f"slope = {slope:.2f}\n"
                f"MBE = {mbe:.2f}\n"
                f"MAE = {mae:.2f}\n"
                f"RMSE = {rmse:.2f}\n"
                f"R2 = {r2:.2f}\n"
            ),
            transform=ax.transAxes,
            fontsize=10,
            ha="right",
            va="bottom"
        )

        ax.grid(alpha=0.3)

    for j in range(n_groups, len(axes)):
        fig.delaxes(axes[j])
    
    fig.suptitle(f"Measured vs estimated {variable.upper()}", fontsize=16)
    fig.tight_layout()
    plt.show()

In [ ]:
plot_metrics(results,variable="le")

In [ ]:
def plot_metrics_plotly(df: pd.DataFrame, variable: str = "le"):
    """
    Interactive plot with Plotly (hover shows date)
    """

    groups = list(df.name.unique())
    groups.append("all")
    n_groups = len(groups)

    # assign one color per group (excluding "all")
    base_groups = df.name.unique()
    colors = px.colors.qualitative.Plotly

    color_map = {
    g: colors[i % len(colors)]
    for i, g in enumerate(base_groups)
    }

    ncols = 3
    nrows = int(np.ceil(n_groups / ncols))

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=groups,
        horizontal_spacing=0.06,
    vertical_spacing=0.14
    )

    for i, group in enumerate(groups):
        row = i // ncols + 1
        col = i % ncols + 1

        if group == "all":
            subset = df
        else:
            subset = df[df.name == group]

        measured = subset[f"ec_{variable}"]
        estimated = subset[variable]

        if group == "all":
            for g in df.name.unique():
                subset = df[df.name == g]

                fig.add_trace(
                    go.Scatter(
                        x=subset[f"ec_{variable}"],
                        y=subset[variable],
                        mode="markers",
                        marker=dict(
                            size=6,
                            opacity=0.5,
                            color=color_map[g]
                        ),
                        name=g,
                        legendgroup=g,
                        showlegend=(group=="all"),  # only show legend here if you want
                        customdata=subset["date"],
                        hovertemplate=(
                            "Group: " + g + "<br>"
                            "Measured: %{x:.3f}<br>"
                            "Estimated: %{y:.3f}<br>"
                            "Date: %{customdata}<extra></extra>"
                        )
                    ),
                    row=row,
                    col=col
                )
        else:
            # Scatter with hover info (including date)
            fig.add_trace(
                go.Scatter(
                    x=measured,
                    y=estimated,
                    mode="markers",
                    marker=dict(size=6, opacity=0.6,color=color_map.get(group, "gray")),
                    name=group,
                    showlegend=False,
                    customdata=subset["date"],
                    hovertemplate=(
                        "Measured: %{x:.2f}<br>"
                        "Estimated: %{y:.2f}<br>"
                        "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                    )
                ),
                row=row,
                col=col
            )

        # Identity line
        lims = [
            min(measured.min(), estimated.min()),
            max(measured.max(), estimated.max())
        ]

        fig.add_trace(
            go.Scatter(
                x=lims,
                y=lims,
                mode="lines",
                line=dict(dash="dash", color="black"),
                showlegend=False
            ),
            row=row,
            col=col
        )

        # Linear fit
        slope, intercept = np.polyfit(measured, estimated, 1)

        fig.add_trace(
            go.Scatter(
                x=[0, measured.max()],
                y=[intercept, measured.max() * slope + intercept],
                mode="lines",
                line=dict(color="red"),
                showlegend=False
            ),
            row=row,
            col=col
        )

        # Metrics
        slope_m, mbe, mae, rmse, r2 = compute_metrics(measured, estimated)

        fig.add_annotation(
            x=0.97,
            y=0.03,
            xref="x domain",
            yref="y domain",
            text=(
                f"slope = {slope_m:.2f}<br>"
                f"MBE = {mbe:.2f}<br>"
                f"MAE = {mae:.2f}<br>"
                f"RMSE = {rmse:.2f}<br>"
                f"R2 = {r2:.2f}"
            ),
            showarrow=False,
            align="right",
            row=row,
            col=col,
        )

    fig.update_xaxes(title_text=f"Measured {variable.upper()}")
    fig.update_yaxes(title_text=f"Estimated {variable.upper()}")
    fig.update_layout(
        height=400 * nrows,
        width=500 * ncols,
        title=f"Measured vs Estimated {variable.upper()}",
    )

    fig.show()

In [ ]:
plot_metrics_plotly(results,variable="le")

In [ ]:
plot_metrics_plotly(results,variable="ef")

## Focus on one point

In [ ]:
def search(df: pd.DataFrame, name:str,date:str, remove_key: list[str]|None=None) -> list[dict]:
    """ 
    Search in dataframe
    """ 
    date_dt = dt.datetime.strptime(date,"%Y-%m-%d")
    query = f"name.str.contains('{name}') and date >= '{date_dt:%Y-%m-%d} 00:00:00-07:00' and date<'{date_dt+dt.timedelta(days=1):%Y-%m-%d} 00:00:00-07:00' "
    res = df.query(query)
    if len(res)>1:
        print(f"Nb resuls= {len(res)}")
    if len(res)==0:
        print("Data not found")
        return [{}]
    res = res.to_dict('records')
    if remove_key is not None:
        for d in res:
            deque(map(d.pop, remove_key, repeat(None)), 0)
    return res

def convert_to_old_stic(params:dict):
    for k,v in pd.DataFrame.from_records([params]).rename(columns={"ts":"TS","ta":"TA","td":"TD","rh":"RH","lai":"LAI","rn":"RN","ln":"Lnet","local_time":"ttSEC"}).to_dict("records")[0].items():
        print(f"{k}=np.array([{v}]),")

def focus_one_point(input_df:pd.DataFrame, results_df:pd.DataFrame, name:str, date:str):
    res = search(results_df,name,date)[0]
    print(f"Date: {res["date"]:%Y-%m-%d}")
    print(f"LE (estimaded) = {res["le"]:.3f}")
    print(f"LE (measured) = {res["ec_le"]:.3f}")
    params = search(input_df,name,date,remove_key=["name","date","date (utc)","lat","lon"])[0]
    print("\nParameters:")
    for k,v in params.items():
        print(f"{k}={v:.6f}")
    print("\nConversion to old version of STIC")
    convert_to_old_stic(params)
    print("\nDetail STIC execution :")
    le, ef, converged = stic.run_stic_model_pixel(**params, threshold=0.01, nb_steps=15, debug=True)
    print(f"LE (computed) = {le:.3f}")



In [ ]:
focus_one_point(data,results,name="Bi2",date="2021-04-06")

In [ ]:
plot_data(pd.DataFrame({
        "date": results["date"],
        "name": results["name"],
        "error": results["le"].values-results["ec_le"].values,
    }),variable="error")

In [ ]:
def plot_correlation(data_df: pd.DataFrame, results_df: pd.DataFrame, method:str="pearson"):
    """
    Plot correlation matrix
    """
    # Merge data
    analysis = pd.merge(left=pd.DataFrame({
        "date": results_df["date"],
        "name": results_df["name"],
        "error": results_df["le"].values-results_df["ec_le"].values,
    }),
    right=data_df[["date","name","ts","ta","td","rh","fc","lai","rn","ln"]],
    how="inner",on=["date","name"])
    # Compute the correlation matrix
    corr = analysis[["error","ts","ta","td","rh","fc","lai","rn","ln"]].corr(method=method)
    # Keep only upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool))
    corr = corr.where(mask)
    # Plot the correlation matrix
    fig = px.imshow(corr,text_auto=".2f")
    # Print value
    fig.update_traces(
    hovertemplate=
    "<b>Variables:</b> %{x} vs %{y}<br>" +
    "<b>Correlation:</b> %{z:.2f}<br>" +
    "<extra></extra>",
    hoverongaps=False
)

    # Format x-axis as YYYY-MM-DD
    fig.update_xaxes(
        side="top",
        showgrid=False
    )
    fig.update_yaxes(showgrid=False)

    # Improve layout
    fig.update_layout(
        width=800,
        height=700,
        title={
        "text": f"Correlation {method}",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 20}
       },
       coloraxis_colorbar={
        "x":1.02,   # move closer (default is ~1.05–1.1)
        "len":0.9   # optional: control height
       }
    )

    fig.show()


In [ ]:
plot_correlation(data,results,method="pearson")

In [ ]:
plot_correlation(data,results,method="kendall")

## Discussion

* LE is generally underestimated compared to measured values across all sites

* There is outliers in input data:

    * LST very low compared to air temperature. For instance 2020-09-26, LST = 2.3°C and Ta = 23.8°C. Maybe it is due to a cloud.


Questions:

* How a negative EC measured LE must be interpreted?
* Should the input data be filtered? If so, based on what criteria?
* Should the EC data be filtered? If so, based on what criteria?

In [ ]:
assert False

## Extrapolate to daily

In [ ]:
def compute_ratio(date,lon,lat):
    """
    Compute ratio
    """
    # Compute TOA solar radiation
    toa_inst = compute_toa_solar_radiation(
        date,
        lon,#x coords
        lat,#y coords
        "epsg:4326",
    )
    toa_daily = compute_daily_toa_solar_radiation(
        date,
        lon,
        lat,
        "epsg:4326",
    )
    return toa_daily / toa_inst


In [ ]:
%%time
ratio = results.apply(lambda x:compute_ratio(x["date"].to_pydatetime(),x["lon"],x["lat"]),axis=1)

In [ ]:
results["daily_et"] = results["et"] * ratio
results["daily_le"] = results["le"] * ratio

In [ ]:
results.head(20)